<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 8


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

банковские карты
#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public class CreditCard
{
    public string Number { get; set; }
    public string Owner { get; set; }
    public string ExpiryDate { get; set; } 
    public string CVV { get; set; }
    public string Currency { get; set; }
    public string BankName { get; set; }
    public string CardType { get; set; }
    public DateTime IssueDate { get; set; }
    protected decimal Money { get; set; }
    public bool IsBlocked { get; private set; }
    protected int TransactionCount { get; set; }

    // Конструкторы (перегрузка)
    public CreditCard(string number, string owner, string expiryDate, string cvv, decimal money = 0)
        : this(number, owner, expiryDate, cvv, "RUB", "Банк", "Standard", money)
    {
    }

    public CreditCard(string number, string owner, string expiryDate, string cvv, string currency, 
                     string bankName, string cardType, decimal money = 0)
    {
        Number = number;
        Owner = owner;
        ExpiryDate = expiryDate;
        CVV = cvv;
        Currency = currency;
        BankName = bankName;
        CardType = cardType;
        Money = money;
        IsBlocked = false;
        IssueDate = DateTime.Now;
        TransactionCount = 0;
    }

    // Методы GetInfo (перегрузка)
    public virtual string GetInfo()
    {
        return $"Карта: {Number} Владелец: {Owner} Срок: {ExpiryDate} CVV: {CVV} Баланс: {Money} {Currency} Статус: {(IsBlocked ? "Заблокирована" : "Активна")}";
    }

    public virtual string GetInfo(bool detailed)
    {
        if (detailed)
        {
            return GetInfo() + $" Банк: {BankName} Тип: {CardType} Выпуск: {IssueDate:dd.MM.yyyy}";
        }
        return GetInfo();
    }

    // Методы Pay (перегрузка)
    public virtual string Pay(decimal amount)
    {
        return Pay(amount, "Онлайн покупка");
    }

    public virtual string Pay(decimal amount, string description)
    {
        if (IsBlocked)
            return "Карта заблокирована! Оплата невозможна.";

        if (Money >= amount)
        {
            Money -= amount;
            TransactionCount++;
            return $"Оплата {amount} {Currency} ({description}). Успешно! Осталось: {Money} {Currency}. Транзакция #{TransactionCount}";
        }
        else
        {
            return $"Не хватает денег! Нужно: {amount} {Currency}, есть: {Money} {Currency}";
        }
    }

    // Методы TransferMoney (перегрузка)
    public virtual string TransferMoney(CreditCard otherCard, decimal amount)
    {
        return TransferMoney(otherCard, amount, "Перевод между картами");
    }

    public virtual string TransferMoney(CreditCard otherCard, decimal amount, string description)
    {
        if (IsBlocked)
            return "Карта заблокирована! Перевод невозможен.";

        if (Money >= amount)
        {
            Money -= amount;
            otherCard.Money += amount;
            TransactionCount++;
            return $"Перевод {amount} {Currency} на карту {otherCard.Number} ({description}) выполнен!";
        }
        else
        {
            return $"Не хватает денег для перевода!";
        }
    }

    // Новые методы
    public virtual void Deposit(decimal amount)
    {
        Deposit(amount, "Пополнение счета");
    }

    public virtual void Deposit(decimal amount, string description)
    {
        Money += amount;
        Console.WriteLine($"Пополнение: {amount} {Currency}. Примечание: {description}");
    }

    public virtual string CheckBalance()
    {
        return $"Баланс: {Money} {Currency}";
    }

    public virtual string CheckBalance(string format)
    {
        return format.ToLower() switch
        {
            "full" => $"Карта {Number}: {Money} {Currency}",
            "short" => $"{Money} {Currency}",
            _ => Money.ToString()
        };
    }

    public void BlockCard()
    {
        IsBlocked = true;
    }

    public void UnblockCard()
    {
        IsBlocked = false;
    }

    public int GetTransactionCount()
    {
        return TransactionCount;
    }
}

public class GoldCard : CreditCard
{
    public int Bonuses { get; private set; }
    public decimal CashbackRate { get; set; }
    public int CreditLimit { get; set; }
    public bool TravelInsurance { get; set; }

    // Конструкторы (перегрузка)
    public GoldCard(string number, string owner, string expiryDate, string cvv, decimal money = 0)
        : base(number, owner, expiryDate, cvv, money)
    {
        Bonuses = 0;
        CashbackRate = 0.05m;
        CreditLimit = 50000;
        TravelInsurance = true;
        CardType = "Gold";
    }

    public GoldCard(string number, string owner, string expiryDate, string cvv, decimal cashbackRate, 
                   int creditLimit, decimal money = 0)
        : this(number, owner, expiryDate, cvv, money)
    {
        CashbackRate = cashbackRate;
        CreditLimit = creditLimit;
    }

    // Переопределение методов (перекрытие)
    public override string Pay(decimal amount)
    {
        return Pay(amount, "Онлайн покупка");
    }

    public override string Pay(decimal amount, string description)
    {
        if (IsBlocked)
            return "Карта заблокирована! Оплата невозможна.";

        // Учет кредитного лимита
        if (Money + CreditLimit >= amount)
        {
            string result;
            if (Money >= amount)
            {
                Money -= amount;
                result = $"Оплата {amount} {Currency} ({description}). Успешно!";
            }
            else
            {
                decimal creditUsed = amount - Money;
                Money = 0;
                result = $"Оплата {amount} {Currency} ({description}). Использован кредит: {creditUsed} {Currency}";
            }

            // Начисление бонусов и кэшбэка
            int newBonuses = (int)(amount / 100);
            Bonuses += newBonuses;
            decimal cashback = amount * CashbackRate;
            Money += cashback;
            
            TransactionCount++;
            return result + $" Начислено бонусов: {newBonuses} Кэшбэк: {cashback} {Currency}. Транзакция #{TransactionCount}";
        }
        else
        {
            return $"Не хватает денег с учетом кредитного лимита!";
        }
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" Бонусы: {Bonuses} Кэшбэк: {CashbackRate * 100}% Кредитный лимит: {CreditLimit} {Currency}";
    }

    public override string GetInfo(bool detailed)
    {
        if (detailed)
        {
            return GetInfo() + $" Страхование путешествий: {(TravelInsurance ? "Да" : "Нет")}";
        }
        return GetInfo();
    }

    // Новые методы
    public void RedeemBonuses(decimal amount)
    {
        RedeemBonuses((int)amount, "Обмен бонусов");
    }

    public void RedeemBonuses(int bonuses, string reason)
    {
        if (Bonuses >= bonuses)
        {
            Bonuses -= bonuses;
            decimal bonusAmount = bonuses * 0.1m;
            Money += bonusAmount;
            Console.WriteLine($"Использовано {bonuses} бонусов ({reason}). Зачислено: {bonusAmount} {Currency}");
        }
    }

    public string GetInsuranceInfo()
    {
        return TravelInsurance ? "Страхование путешествий активно" : "Страхование путешествий неактивно";
    }
}

public class PremiumCard : CreditCard
{
    public string Support { get; set; }
    public string InsuranceNumber { get; set; }
    public decimal InvestmentLimit { get; set; }
    public bool LoungeAccess { get; set; }

    public PremiumCard(string number, string owner, string expiryDate, string cvv, string support, 
                      string insuranceNumber, decimal money = 0)
        : base(number, owner, expiryDate, cvv, money)
    {
        Support = support;
        InsuranceNumber = insuranceNumber;
        InvestmentLimit = 100000;
        LoungeAccess = true;
        CardType = "Premium";
    }

    // Перегрузка конструктора
    public PremiumCard(string number, string owner, string expiryDate, string cvv, string support,
                      string insuranceNumber, decimal investmentLimit, bool loungeAccess, decimal money = 0)
        : this(number, owner, expiryDate, cvv, support, insuranceNumber, money)
    {
        InvestmentLimit = investmentLimit;
        LoungeAccess = loungeAccess;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" Поддержка: {Support} Страховой номер: {InsuranceNumber}";
    }

    public override string GetInfo(bool detailed)
    {
        if (detailed)
        {
            return GetInfo() + $" Лимит инвестиций: {InvestmentLimit} {Currency} Доступ в лаунж: {(LoungeAccess ? "Да" : "Нет")}";
        }
        return GetInfo();
    }

    // Переопределение метода Deposit
    public override void Deposit(decimal amount, string description)
    {
        base.Deposit(amount, description);
        // Премиум-бонус при пополнении
        decimal bonus = amount * 0.01m;
        Money += bonus;
        Console.WriteLine($"Премиум-бонус: {bonus} {Currency}");
    }

    public string GetInsuranceInfo()
    {
        return $"Страховой номер: {InsuranceNumber}. Страховка действует до {ExpiryDate}.";
    }

    public string RequestInvestment(decimal amount)
    {
        return amount <= InvestmentLimit 
            ? $"Инвестиция на {amount} {Currency} одобрена" 
            : $"Превышен лимит инвестиций {InvestmentLimit} {Currency}";
    }
}

public class PlatinumCard : GoldCard
{
    public string ConciergeService { get; set; }
    public bool GlobalAssistance { get; set; }
    public string PriorityPass { get; set; }

    public PlatinumCard(string number, string owner, string expiryDate, string cvv, string conciergeService, decimal money = 0)
        : base(number, owner, expiryDate, cvv, money)
    {
        ConciergeService = conciergeService;
        CashbackRate = 0.1m;
        CreditLimit = 100000;
        GlobalAssistance = true;
        PriorityPass = "PLATINUM-001";
        CardType = "Platinum";
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" Консьерж-сервис: {ConciergeService}";
    }

    public override string GetInfo(bool detailed)
    {
        if (detailed)
        {
            return GetInfo() + $" Глобальная помощь: {(GlobalAssistance ? "Да" : "Нет")} Priority Pass: {PriorityPass}";
        }
        return GetInfo();
    }

    // Перегрузка метода Pay
    public string Pay(decimal amount, string merchant, string location)
    {
        string result = Pay(amount, $"Покупка в {merchant} ({location})");
        return result + " (Platinum привилегия)";
    }

    public string CallConcierge()
    {
        return $"Консьерж-сервис {ConciergeService} вызван!";
    }

    public string CallConcierge(string request)
    {
        return $"Консьерж-сервис {ConciergeService}: '{request}' - запрос принят!";
    }
}

public interface ILoyaltyProgram
{
    void AddLoyaltyPoints(int points);
    void AddLoyaltyPoints(int points, string reason);
    int GetLoyaltyPoints();
    string GetLoyaltyStatus();
}

public class LoyaltyGoldCard : GoldCard, ILoyaltyProgram
{
    private int LoyaltyPoints { get; set; }
    public string LoyaltyLevel { get; set; }

    public LoyaltyGoldCard(string number, string owner, string expiryDate, string cvv, decimal money = 0)
        : base(number, owner, expiryDate, cvv, money)
    {
        LoyaltyPoints = 0;
        LoyaltyLevel = "Bronze";
    }

    // Реализация интерфейса
    public void AddLoyaltyPoints(int points)
    {
        AddLoyaltyPoints(points, "Стандартное начисление");
    }

    public void AddLoyaltyPoints(int points, string reason)
    {
        LoyaltyPoints += points;
        UpdateLoyaltyLevel();
        Console.WriteLine($"Начислено {points} баллов лояльности. Причина: {reason}");
    }

    public int GetLoyaltyPoints()
    {
        return LoyaltyPoints;
    }

    public string GetLoyaltyStatus()
    {
        return $"Уровень: {LoyaltyLevel} Баллы: {LoyaltyPoints}";
    }

    private void UpdateLoyaltyLevel()
    {
        LoyaltyLevel = LoyaltyPoints switch
        {
            < 1000 => "Bronze",
            < 5000 => "Silver", 
            < 10000 => "Gold",
            _ => "Platinum"
        };
    }

    // Переопределение метода Pay с учетом лояльности
    public override string Pay(decimal amount, string description)
    {
        string result = base.Pay(amount, description);
        if (result.Contains("Успешно") || result.Contains("Использован кредит"))
        {
            int loyaltyPoints = (int)(amount / 50);
            AddLoyaltyPoints(loyaltyPoints, $"Покупка: {description}");
        }
        return result;
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $" {GetLoyaltyStatus()}";
    }
}

// Generic класс для работы с коллекцией карт
public class CardCollection<T> where T : CreditCard
{
    private List<T> _cards = new List<T>();
    public string CollectionName { get; set; }

    public CardCollection(string name)
    {
        CollectionName = name;
    }

    public void AddCard(T card)
    {
        _cards.Add(card);
        Console.WriteLine($"Карта {card.Number} добавлена в коллекцию '{CollectionName}'");
    }

    public void RemoveCard(T card)
    {
        _cards.Remove(card);
        Console.WriteLine($"Карта {card.Number} удалена из коллекции");
    }

    public void DisplayAllCards()
    {
        Console.WriteLine($"\n=== КОЛЛЕКЦИЯ '{CollectionName}' ({_cards.Count} карт) ===");
        foreach (var card in _cards)
        {
            Console.WriteLine(card.GetInfo(true));
            Console.WriteLine("---");
        }
    }

    public T FindCardByNumber(string number)
    {
        return _cards.Find(card => card.Number == number);
    }

    public void ProcessPayments(decimal amount)
    {
        Console.WriteLine($"\n=== МАССОВЫЕ ОПЛАТЫ ({amount}) ===");
        foreach (var card in _cards)
        {
            Console.WriteLine($"{card.Owner}: {card.Pay(amount)}");
        }
    }

    public decimal GetTotalBalance()
    {
        decimal total = 0;
        foreach (var card in _cards)
        {
            // Используем reflection для доступа к protected полю Money
            var moneyField = card.GetType().GetField("Money", System.Reflection.BindingFlags.NonPublic | System.Reflection.BindingFlags.Instance);
            if (moneyField != null)
            {
                total += (decimal)moneyField.GetValue(card);
            }
        }
        return total;
    }
}

        Console.WriteLine("=== РАСШИРЕННАЯ СИСТЕМА КРЕДИТНЫХ КАРТ ===\n");

        // Создание карт
        CreditCard ivanCard = new CreditCard("1111-2222", "Иван", "12/25", "123", 5000);
        GoldCard petrCard = new GoldCard("3333-4444", "Петр", "06/26", "456", 0.07m, 75000, 10000);
        PremiumCard mariaCard = new PremiumCard("5555-6666", "Мария", "09/27", "789", "8-800-555", "INS-12345", 500);
        PlatinumCard alexCard = new PlatinumCard("7777-8888", "Алексей", "01/28", "000", "VIP-Concierge", 20000);
        LoyaltyGoldCard olgaCard = new LoyaltyGoldCard("9999-0000", "Ольга", "03/29", "111", 15000);

        // Демонстрация перегрузки методов
        Console.WriteLine("ИНФОРМАЦИЯ О КАРТАХ (подробно):");
        Console.WriteLine(ivanCard.GetInfo(true));
        Console.WriteLine(petrCard.GetInfo(true));
        Console.WriteLine(mariaCard.GetInfo(true));
        Console.WriteLine(alexCard.GetInfo(true));
        Console.WriteLine(olgaCard.GetInfo(true));

        Console.WriteLine("\nОПЕРАЦИИ С КАРТАМИ:");
        
        // Перегрузка методов Pay
        Console.WriteLine("Иван оплачивает покупку:");
        Console.WriteLine(ivanCard.Pay(2000));
        Console.WriteLine(ivanCard.Pay(1500, "Ресторан"));

        // Перегрузка методов Transfer
        Console.WriteLine("\nПетр переводит деньги:");
        Console.WriteLine(petrCard.TransferMoney(mariaCard, 3000));
        Console.WriteLine(petrCard.TransferMoney(mariaCard, 2000, "Подарок"));

        // Перегрузка в PlatinumCard
        Console.WriteLine("\nАлексей использует платиновые услуги:");
        Console.WriteLine(alexCard.Pay(5000, "Бутик", "Париж"));
        Console.WriteLine(alexCard.CallConcierge());
        Console.WriteLine(alexCard.CallConcierge("Забронировать столик в ресторане"));

        // Работа с интерфейсом
        Console.WriteLine("\nОльга и бонусы лояльности:");
        olgaCard.AddLoyaltyPoints(500, "Регистрация");
        Console.WriteLine(olgaCard.Pay(7000, "Техника"));
        Console.WriteLine(olgaCard.GetLoyaltyStatus());

        // Использование generic коллекции
        Console.WriteLine("\n" + new string('=', 50));
        CardCollection<CreditCard> allCards = new CardCollection<CreditCard>("Все карты");
        allCards.AddCard(ivanCard);
        allCards.AddCard(petrCard);
        allCards.AddCard(mariaCard);
        allCards.AddCard(alexCard);
        allCards.AddCard(olgaCard);

        allCards.DisplayAllCards();
        allCards.ProcessPayments(1000);

        // Специализированная коллекция
        CardCollection<GoldCard> goldCards = new CardCollection<GoldCard>("Золотые карты");
        goldCards.AddCard(petrCard);
        goldCards.AddCard(olgaCard);
        goldCards.DisplayAllCards();

=== РАСШИРЕННАЯ СИСТЕМА КРЕДИТНЫХ КАРТ ===

ИНФОРМАЦИЯ О КАРТАХ (подробно):
Карта: 1111-2222 Владелец: Иван Срок: 12/25 CVV: 123 Баланс: 5000 RUB Статус: Активна Банк: Банк Тип: Standard Выпуск: 15.10.2025
Карта: 3333-4444 Владелец: Петр Срок: 06/26 CVV: 456 Баланс: 10000 RUB Статус: Активна Бонусы: 0 Кэшбэк: 7,00% Кредитный лимит: 75000 RUB Страхование путешествий: Да
Карта: 5555-6666 Владелец: Мария Срок: 09/27 CVV: 789 Баланс: 500 RUB Статус: Активна Поддержка: 8-800-555 Страховой номер: INS-12345 Лимит инвестиций: 100000 RUB Доступ в лаунж: Да
Карта: 7777-8888 Владелец: Алексей Срок: 01/28 CVV: 000 Баланс: 20000 RUB Статус: Активна Бонусы: 0 Кэшбэк: 10,0% Кредитный лимит: 100000 RUB Консьерж-сервис: VIP-Concierge Глобальная помощь: Да Priority Pass: PLATINUM-001
Карта: 9999-0000 Владелец: Ольга Срок: 03/29 CVV: 111 Баланс: 15000 RUB Статус: Активна Бонусы: 0 Кэшбэк: 5,00% Кредитный лимит: 50000 RUB Уровень: Bronze Баллы: 0 Страхование путешествий: Да

ОПЕРАЦИИ С КАРТАМИ:
Иван оплач